In [5]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
from pathlib import Path
# Procura a raiz do projeto (pasta que contém `src`) subindo na arvore
def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'src').is_dir():
            return p
    return start
proj_root = str(find_project_root())
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)
print('Project root added to sys.path:', proj_root)

from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import LabelBinarizer, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict, cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from src.preprocessing import SavgolFilter, SNV, make_savgol, make_scaler, make_snv
from src.utils import extrair_numero
from src.models import PLSDAClassifier, PLSDAMulticlass

Project root added to sys.path: c:\Users\Pedro\Downloads\LIBS_NEW


In [ ]:
import os

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold

from src.training import run_experiment
from src.utils import extrair_numero


# =========================================================
# ESCOLHA DO LASER
# =========================================================

laser = int(input('Qual laser utilizado? 266, 532 ou 1064\n'))

if laser not in [532, 1064, 266]:
    raise ValueError('O valor deve ser 266, 532 ou 1064')


# =========================================================
# LEITURA DO DATASET
# =========================================================

print('\nLendo dataset...')
file_path = Path(proj_root) / f'dataset/laser{laser}.dat'

df = pd.read_csv(file_path, sep=';', header=None)

y_raw = df.iloc[0, 1:].values.astype(str)
X_full = df.iloc[1:, 1:].values.astype(float).T


# =========================================================
# SEPARAÇÃO DAS CLASSES
# =========================================================

extra_a_labels_266 = {
    'Pilao_sample',
    'Evolutto_sample',
    'Do_Ponto_sample',
    'America_sample',
    'Coamo_sample',
}

X = []
y = []
groups = []

for xi, yi in zip(X_full, y_raw):
    num = extrair_numero(yi)
    if laser == 266:
        if (yi.startswith('A') and num is not None and num <= 12) or yi in extra_a_labels_266:
            X.append(xi)
            y.append('A')
            groups.append(str(num) if num is not None else yi)
        elif yi.startswith('R') and num is not None and num <= 9:
            X.append(xi)
            y.append('R')
            groups.append(str(num))
    elif laser == 532:
        if yi.startswith('A') and num is not None and num <= 8:
            X.append(xi)
            y.append('A')
            groups.append(str(num))
        elif yi.startswith('R') and num is not None and num <= 8:
            X.append(xi)
            y.append('R')
            groups.append(str(num))
    else:
        if yi.startswith('A') and num is not None and num <= 12:
            X.append(xi)
            y.append('A')
            groups.append(str(num))
        elif yi.startswith('R') and num is not None and num <= 9:
            X.append(xi)
            y.append('R')
            groups.append(str(num))

X = np.array(X)
y = np.array(y)
groups = np.array(groups)

print(f'Formato do dataset: {X.shape}')
print(f'Labels: {np.unique(y)}')
print(f'Grupos: {np.unique(groups)}')

unique_groups = np.unique(groups)
if len(unique_groups) < 2:
    raise ValueError('É necessário pelo menos 2 grupos para usar StratifiedGroupKFold.')

n_splits = min(8, len(unique_groups))
sgkf = StratifiedGroupKFold(n_splits=n_splits)


# =========================================================
# CONFIGURAÇÃO DOS EXPERIMENTOS
# =========================================================

preprocessamentos = {
    'none': [],
    'snv': [make_snv()],
    'savgol_snv': [make_savgol(), make_snv()],
}

modelos = {
    'svm_linear': SVC(kernel='linear', gamma='scale'),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'pls_da': PLSDAClassifier(n_components=2),
}


# =========================================================
# PASTAS DE SAÍDA
# =========================================================

os.makedirs(f'models_groupkfold/laser{laser}', exist_ok=True)
os.makedirs(f'plots_groupkfold/laser{laser}', exist_ok=True)


# =========================================================
# TREINAMENTO E AVALIAÇÃO
# =========================================================

print('Executando Cross Validation com StratifiedGroupKFold...')
results_data = run_experiment(
    X=X,
    y=y,
    preprocessamentos=preprocessamentos,
    modelos=modelos,
    cv=sgkf,
    output_models_dir=f'models_groupkfold/laser{laser}',
    output_plots_dir=f'plots_groupkfold/laser{laser}',
    laser=laser,
    labels=np.unique(y),
    experiment_name='stratified_groupkfold_binary',
    display_labels=['Arabica', 'Robusta'],
    groups=groups,
    summary_filename='all_pipelines_summary.json',
    ranking_filename='ranking_final_groupkfold.png',
)

results = results_data['results']
print('==============================')
print('RANKING FINAL')
print('==============================')

for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True):
    print(f'{k}: {v:.4f}')

print(f"Resumo consolidado salvo em: {results_data['summary_path']}")
print(f"Plot final salvo em: {results_data['ranking_path']}")
print('Processo concluído!')


Lendo dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'dataset/laser266.dat'